# 지역화폐 뉴스 EDA

- 대상: 제공 데이터 `05_공공정책_키워드_지역화폐` (36,888건)
- 목적: 벡터 DB 구성 전 문서의 구성·기간·길이·품질·중복 실태 파악, 전처리 규칙과 임계값 근거 확보
- 결과 수치는 `data/interim/eda_summary.json`으로 저장하여 리포트 2항·4항에 인용

In [1]:
import os, re, json, hashlib, unicodedata
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'notebook_connected'
pd.set_option('display.max_colwidth', 120)
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
DATA_DIR = os.path.join(ROOT, 'data', 'news_csv_5y')
OUT_DIR = os.path.join(ROOT, 'data', 'interim')
os.makedirs(OUT_DIR, exist_ok=True)
KW = '지역화폐'
SUMMARY = {}

## 1. 적재와 기본 구성

In [2]:
# 파일명이 NFD로 저장된 경우가 있어 NFC 정규화 후 매칭
files = [f for f in os.listdir(DATA_DIR)
         if unicodedata.normalize('NFC', f).startswith('05_공공정책_키워드_지역화폐')]
assert len(files) == 1, files
df = pd.read_csv(os.path.join(DATA_DIR, files[0]), encoding='utf-8-sig', dtype=str).fillna('')
df['date'] = pd.to_datetime(df['발행일'], errors='coerce')
df['year'] = df['date'].dt.year
df['ym'] = df['date'].dt.to_period('M').astype(str)
df['body_len'] = df['본문전문'].str.len()
df['title_len'] = df['제목'].str.len()
print(df.shape)
print('날짜 파싱 실패:', df['date'].isna().sum())
print('빈 본문:', (df['body_len'] == 0).sum(), ' 빈 제목:', (df['title_len'] == 0).sum())
SUMMARY['raw_rows'] = int(len(df))
SUMMARY['empty_body'] = int((df['body_len'] == 0).sum())
df.head(3)

(36888, 12)
날짜 파싱 실패: 0
빈 본문: 41  빈 제목: 0


,제목,본문전문,원문URL,발행일,매체명,문서종류,수집키워드,date,year,ym,body_len,title_len
0,"여야, '지역화폐' 두고 기싸움…골목 경제 살리기 VS 지역화폐 효과 글쎄","용감한 토크쇼 '직설' 손석우 앵커 경제평론가 (건국대 겸임교수), 이인철 참조은경제연구소 소장, 원영섭 변호사 (전 국민의힘 미디어법률단장), 이동학 전 민주당 최고위원 여야, '지역화폐' 두고 기싸...",https://n.news.naver.com/mnews/article/374/0000425060,2025-02-12T14:12:00,SBS Biz,news,지역화폐,2025-02-12 14:12:00,2025,2025-02,451,41
1,최병일 안양시의장 ‘지역화폐 활성화’ 챌린지 동참,최병일 안양시의회 의장이 8일 민생 안정과 지역경제 활성화를 위해 '지역화폐 사용 함께해요 릴레이 챌린지'(이하 지역화폐 챌린지)에 동참했다. 지역화폐 챌린지는 지속적인 정부 보조금 지원을 촉구하고자 지역...,https://n.news.naver.com/mnews/article/014/0004896407?sid=102,2022-09-09T07:49:00,파이낸셜뉴스,news,지역화폐,2022-09-09 07:49:00,2022,2022-09,706,27
2,[이슈& 직설] ‘재난지원금’ 빠지고 ‘지역화폐’…여·야 예산 쟁점 되나,"경제 와이드 이슈 '직설' 최양오 현대경제연구원 고문, 김연학 서강대 기술경영전문대학원 초빙교수, 정용건 사회연대포럼 대표 지역화폐 예산 올해 1조522억 원, 내년은 2,403억 민주당 ""지역화...",https://news.naver.com/main/read.naver?mode=LS2D&mid=shm&sid1=101&sid2=263&oid=374&aid=0000265529,2021-11-24T14:05:00,SBS Biz,news,지역화폐,2021-11-24 14:05:00,2021,2021-11,341,40


## 2. 기간 분포

In [3]:
by_year = df['year'].value_counts().sort_index()
print(by_year.to_string())
SUMMARY['by_year'] = {int(k): int(v) for k, v in by_year.items()}
by_month = df.groupby('ym').size().reset_index(name='n')
fig = px.line(by_month, x='ym', y='n', title='월별 기사 수 (지역화폐)')
fig.show()

year
2021     2622
2022     6986
2023     4471
2024     5304
2025    10694
2026     6811


In [4]:
# 상위 피크 월과 해당 월의 무작위 제목 (시점형 질문 소재)
peaks = by_month.sort_values('n', ascending=False).head(8)
SUMMARY['peak_months'] = dict(zip(peaks['ym'], peaks['n'].astype(int)))
for ym in peaks['ym']:
    sub = df[df['ym'] == ym]
    print(f'== {ym} ({len(sub)}건)')
    for t in sub.sample(min(6, len(sub)), random_state=0)['제목']:
        print('   -', t[:90])

== 2025-06 (1911건)
   - KB, 소상공인 살리기 동참…전통시장 특화카드 선뵌다
   - 민생지원금 모두에게 주되 어려운 계층엔 더 많이 지원
   - 이재명 대통령 지방에 더 투자해야…소비쿠폰도 수도권 멀수록 시범 지원
   - 경기도 지역화폐, 카카오페이로도 간편 결제
   - 지역화폐 도입 속속…지역 경제 마중물 될까?
   - 부산시, 동백전 캐시백 확대 8월까지 연장… 지역소비 총력
== 2025-07 (1508건)
   - 폭염·장마에 발길 끊겨…7월 소상공인 경기전망 '뚝'
   - 소비쿠폰, 첫 주 국민 72% 신청… 6조 5703억 지급
   - 지역화폐株, '31조 추경안' 통과에 강세…코나아이 8%↑[특징주]
   - 경북 영양군서 지역화폐 불법 소각 적발…반출 경위 조사
   - BC카드, 지역화폐 매출 9% 증가...지역 내 선순환 구조 형성
   - 이재준 수원시장, 민생회복소비쿠폰 사용처 스티커 부착
== 2021-11 (1255건)
   - 내년 지역화폐 지원 예산 얼마나 늘릴까… 여·야·정 ‘동상삼몽’
   - 광명시 광명사랑화폐 월 구매한도 100만원으로 상향
   - 지역화폐 농성 현장 방문한 이재명
   - 이재명 “전 국민 소비쿠폰 지급하자”…윤석열에 제안
   - 李 “기재부는 죽어도 안잡혀”… 홍남기에 불편한 심기
   - 평택시, 하나로마트 등 연매출 10억이상 가맹점 지역화폐 사용 제한
== 2025-01 (1082건)
   - 군포시, 지역화폐 인센티브 10%로 상향…충전 한도 월 40만원
   - 민주당, 윤 대통령이 거부한 '지역화폐법' 재발의
   - 민주당 “정부, 지역화폐 발행 등 추경 편성 착수해야” 촉구
   - '추경 논의' 시사한 최상목 국회와 협의 가능…野 일방처리 3법에 '거부권' 행사
   - 김경희 시장 이천시, 분명히 변하고 있다
   - 與, 이재명 '추경 위해 민생지원금 포기' 언급에 꼼수 아니길
== 2026-03 (1061건)
   - 파주시, 11~18세 청소년에 

## 3. 매체 분포

In [5]:
media = df['매체명'].value_counts()
print('매체 수:', media.size)
print(media.head(20).to_string())
WIRE = ['뉴시스', '뉴스1', '연합뉴스']
wire_share = df['매체명'].isin(WIRE).mean()
print(f'통신사 3사 비중: {wire_share:.1%}')
SUMMARY['n_media'] = int(media.size)
SUMMARY['wire_share'] = round(float(wire_share), 4)
SUMMARY['top_media'] = {k: int(v) for k, v in media.head(10).items()}
fig = px.bar(media.head(20)[::-1], orientation='h', title='상위 20개 매체')
fig.show()

매체 수: 96
매체명
뉴시스       3237
뉴스1       2804
연합뉴스      2654
KBS       1463
파이낸셜뉴스    1241
아시아경제     1177
머니투데이     1100
더팩트       1084
헤럴드경제     1048
노컷뉴스      1007
프레시안       983
서울경제       975
이데일리       923
서울신문       909
데일리안       817
세계일보       774
한국경제       663
매일경제       661
아이뉴스24     656
머니S        644
통신사 3사 비중: 23.6%


## 4. 길이 분포

In [6]:
q = df['body_len'].quantile([.01, .05, .25, .5, .75, .95, .99]).astype(int)
print(q.to_string())
for th in [100, 200, 300, 500]:
    print(f'<{th}자: {(df.body_len < th).sum()}건 ({(df.body_len < th).mean():.1%})')
for th in [5000, 10000, 20000]:
    print(f'>{th}자: {(df.body_len > th).sum()}건')
SUMMARY['body_len_quantiles'] = {str(k): int(v) for k, v in q.items()}
SUMMARY['short_lt200'] = int((df.body_len < 200).sum())
SUMMARY['long_gt10000'] = int((df.body_len > 10000).sum())
fig = px.histogram(df[df.body_len < 6000], x='body_len', nbins=120, title='본문 길이 분포 (6,000자 미만)')
fig.show()

0.01     145
0.05     355
0.25     664
0.50     969
0.75    1458
0.95    2745
0.99    6154
<100자: 223건 (0.6%)
<200자: 605건 (1.6%)
<300자: 1351건 (3.7%)
<500자: 4363건 (11.8%)
>5000자: 515건
>10000자: 157건
>20000자: 30건


In [7]:
print('== 200자 미만 샘플')
for _, r in df[df.body_len < 200].sample(8, random_state=1).iterrows():
    print(f'[{r.body_len}] {r["제목"][:60]} || {r["본문전문"][:150]}')
print()
print('== 10,000자 초과 샘플')
for _, r in df[df.body_len > 10000].sample(6, random_state=1).iterrows():
    print(f'[{r.body_len}] {r["매체명"]} | {r["제목"][:70]}')

== 200자 미만 샘플
[97] 이재명, 지역화폐 예산 반드시 살리겠습니다 || 더불어민주당 이재명 대표가 17일 국회 대표실에서 열린 지역화폐 예산확보를 위한 간담회에서 발언하고 있다.    기사를 더 보고 싶다면?          기사 섹션 분류 안내
[74] 화성시, 설 연휴 전후 지역화폐 20% 캐시백 || 제보하기   전화   4444   이메일      카카오톡   '제보' 검색, 채널 추가   네이버, 유튜브에서 뉴스를 구독해주세요!
[98] 이재명, 정부청사 앞 지역화폐·골목상권살리기 운동본부 농성 현장 방문 || 이재명(앞줄 왼쪽 여섯 번째) 더불어민주당 대선 후보가 15일 오후 서울 종로구 정부서울청사 앞에 차려진 지역화폐 골목상권살리기 운동본부 농성 현장을 방문해 발언을 하고 있다.
[150] 광주 북구, 부끄머니 지역 화폐 발행 || 광주 북구 지역화폐 '부끄머니' 발행 첫날인 3일 오전 광주 북구청 광장에 있는 광주은행 출장 버스에서 문인 북구청장이 카드를 발급받고 있다. 부끄머니는 북구 소재 광주은행에서 1인당 최대 50만원까지 구매할 수 있고, 50만원권을 할인받아 41만원에 구입할 수 있다.
[183] [속보] 민주·국힘 ‘13.8조원’ 추경 합의…지역화폐 포함 1.6조원 증액 || 더불어민주당과 국민의힘은 1일 13조8000억 원 규모의 추가경정예산안 처리에 합의했다. 이는 정부가 당초 국회에 제출한 12조2000억 원에서 1조6000억 원 늘어난 것이다. 추경안에는 지역화폐 예산 4000억 원도 편성됐다. 민주당 박찬대 국민의힘 권성동 원내대표
[183] [서산소식] 헌혈할 때마다 지역화폐 1만원 지급 || 충남 서산시는 안정적인 혈액 수급을 위해 헌혈자에게 지역화폐를 지급한다고 29일 밝혔다.      이를 위해 지난해 11월 관련 조례를 개정했다.     시는 오는 8월부터 수혈용 헌혈자에게 1회당 1만원 상당의 지역화폐를 지급할 계획이다.   기사를 더 보고 싶다면?
[187] 지역화폐 ‘울산페이’ 오늘부터 서비스 재개 

[13282] 채널A | 인터뷰 전문…이준석 “보수 진영에서 尹 파문시켜야” [정치시그널]
[12643] 더팩트 | 첫 국회연설 이재명 산업화·민주화 넘어 '기본사회' 30년 준비할 때(전문)
[11219] 오마이뉴스 | 이재명 위기를 기회로, '기본사회'가 답 [교섭단체 대표연설 전문]
[12280] 노컷뉴스 | 김한규 강훈식 퀭한 얼굴… 이제 '일하는 대통령실' 됐다[한판승부]
[12878] YTN | [뉴스앤이슈] 여야, 공천심사 결과 속속 발표...파장은?
[17692] 노컷뉴스 | [당헌 80조] 이재명 야당 탄압 고려 VS 박용진 개정 반대 VS 강훈식 논의는 가능


- 판정 메모: 200자 미만은 사진 설명·제보 안내만 남은 기사가 다수, 단 추경 합의 속보처럼 수치를 담은 단신도 존재
- 10,000자 초과는 시사 토크쇼 녹취·연설 전문이 대부분이며 지역화폐는 부수적 언급

## 5. 관련도 — 키워드 등장 빈도

In [8]:
RELATED = ['지역사랑상품권', '지역상품권', '온누리상품권', '소비쿠폰', '민생회복']
df['kw_title'] = df['제목'].str.contains(KW)
df['kw_cnt'] = df['본문전문'].str.count(KW)
df['lsg_cnt'] = df['본문전문'].str.count('지역사랑상품권')
print('제목에 키워드 포함:', round(df.kw_title.mean(), 3))
print('본문 등장 횟수 분포:')
print(df['kw_cnt'].clip(upper=10).value_counts().sort_index().to_string())
for w in RELATED:
    print(f'{w} 동시 등장 비율: {df["본문전문"].str.contains(w).mean():.1%}')

제목에 키워드 포함: 0.219
본문 등장 횟수 분포:
kw_cnt
0       638
1     19975
2      6896
3      3296
4      1937
5      1127
6       775
7       549
8       409
9       290
10      996
지역사랑상품권 동시 등장 비율: 14.8%


지역상품권 동시 등장 비율: 1.7%
온누리상품권 동시 등장 비율: 3.9%


소비쿠폰 동시 등장 비율: 6.4%
민생회복 동시 등장 비율: 7.9%


In [9]:
# 관련도 필터 후보별 잔존 건수
KEYS = r'지역화폐|지역사랑상품권|지역상품권'
df['key_title'] = df['제목'].str.contains(KEYS)
df['key_cnt'] = df['본문전문'].str.count(KEYS)
df['key_lead'] = df['본문전문'].str[:400].str.contains(KEYS)
df['key_density'] = df['key_cnt'] / (df['body_len'] / 1000).clip(lower=0.3)
cands = {
    'A: 제목 포함': df.kw_title,
    'B: 제목 포함 or 본문>=2': df.kw_title | (df.kw_cnt >= 2),
    'C: 제목 포함 or 본문>=3': df.kw_title | (df.kw_cnt >= 3),
    'R3: 동의어 제목 or 리드400 or (본문>=2 and 1000자당>=1)': df.key_title | df.key_lead | ((df.key_cnt >= 2) & (df.key_density >= 1)),
    'R4: 동의어 제목 or (본문>=2 and 1000자당>=1)': df.key_title | ((df.key_cnt >= 2) & (df.key_density >= 1)),
}
for k, m in cands.items():
    print(f'{k}: {m.sum()}건 ({m.mean():.1%})')
SUMMARY['relevance_candidates'] = {k: int(m.sum()) for k, m in cands.items()}

A: 제목 포함: 8062건 (21.9%)
B: 제목 포함 or 본문>=2: 18515건 (50.2%)
C: 제목 포함 or 본문>=3: 12914건 (35.0%)
R3: 동의어 제목 or 리드400 or (본문>=2 and 1000자당>=1): 26218건 (71.1%)
R4: 동의어 제목 or (본문>=2 and 1000자당>=1): 19322건 (52.4%)


In [10]:
# 필터 B에서 탈락하는 기사(제목 미포함, 본문 1회 등장) 샘플: 노이즈 여부 직접 확인
drop = df[~cands['B: 제목 포함 or 본문>=2']]
print('탈락 건수:', len(drop))
for _, r in drop.sample(12, random_state=2).iterrows():
    body = r['본문전문']
    i = body.find(KW)
    ctx = body[max(0, i - 60): i + 60].replace('\n', ' ')
    print(f'- {r["제목"][:55]} || ...{ctx}...')

탈락 건수: 18373
- 정부 ‘윤석열표 예산’에…야, 상임위 곳곳 증액안 단독 처리 || ...68석의 더불어민주당은 국회 상임위원회 곳곳에서 국민의힘의 반발 속에 연구 개발(R D) 및 지역사랑상품권(지역화폐) 예산 증액안을 단독으로 통과시켰다. 정부가 야당 요구를 외면한 '윤석열표' 예산안을 제출하자, 이...
- 세종시 여민전 7월 1일 발행 확정…중단 위기 넘겼다 || ...중단 위기에 놓였던 세종시 지역화폐 여민전이 다음 달 1일 예정대로 발행되는 것으로 확정됐다. 조상호 세종시장 당선인 인수위원회는 11...
- 전남도, '반값여행' 호응 속 5~6월분 접수 || ...공사가 주관하는 '지역사랑 휴가지원사업'은 인구 감소 지역을 방문하는 관광객에게 여행경비의 50%를 모바일 지역화폐 등으로 환급하는 제도다. 1인 최대 10만 원, 2인 이상은 최대 20만 원을 지원한다. 전남지역 ...
- 전문학, '기본사회본부' 발대…서구형 기본사회 정책 본격 추진 || ... 특히 세부 내용으로 공공임대 확대, 전세보증금 지원, 어린이 돌봄 확대, 돌봄 참여소득 정착, 주민참여형 지역화폐 확산, AI 기반 행정혁신 등을 핵심 과제로 구체화할 방침이다. 앞서 이날 오전 전 후보는 지역 소...
- 부산시의회 제동에 동백전 15% 캐시백 무산 || ...부산 지역화폐 동백전 캐시백 비율을 100일간 기존 10%에서 15%로 올리려던 전재수 시정의 예산 계획에 제동이...
- 매경이 전하는 세상의 지식(매-세-지, 8월 30일) || ...국무회의를 열고 사상 최대 규모인 728조원의 내년도 예산안을 확정했습니다. 인공지능, 연구개발은 물론이고 지역화폐, 아동수당, 농어촌기본소득 등 이재명표 복지사업이 대거 포함됐습니다. 올해 본예산 673조3000억...
- [이시각헤드라인] 12월 20일 라이브투데이1부 || ...여야 이견 여전 여야는 앞서, 오늘 본회의를 열고 예산안을 처리하자고 합의했습니다. 하지만 연구개발 예산과 지역화폐, 새만금 관련 예산 등을 둘러싸

In [11]:
# R3(리드 400자 포함 기준)로 추가 유입되는 기사 샘플
extra = df[cands['R3: 동의어 제목 or 리드400 or (본문>=2 and 1000자당>=1)'] & ~cands['R4: 동의어 제목 or (본문>=2 and 1000자당>=1)']]
print('R3 추가 유입:', len(extra))
for _, r in extra.sample(10, random_state=5).iterrows():
    body = r['본문전문']
    i = re.search(KEYS, body).start()
    print(f'- {r["제목"][:55]} || ...{body[max(0, i - 50): i + 50]}...')

R3 추가 유입: 6896
- 오산시, 공공배달앱 '배달특급' 연말특별이벤트 시행 || ...축하 이미지'를 본인 SNS에 게시하고 인증하거나 이벤트 기간에 배달특급으로 주문할 경우 지역화폐 결제 시 사용 가능한 할인쿠폰을 지급하는 두 가지다. '배달특급 출시 1주년  축...
- 광주 서구, 국회 입법박람회서 우수정책 ‘전국 관심’ || ...목형상점가로 지정한 '골목경제119 프로젝트'를 선보였다. 지자체의 예산 부담이 뒤따르는 지역화폐 대신 전액 국비로 운영되는 온누리상품권 인프라를 적극 활용해 소비 진작 효과와 골...
- [오산소식]시, 오색전 인센티브 유효연장 등 || ...경기 오산시는 행정안전부 '지역화폐 운영 우수지자체' 선정에 따라 지급되는 오색전 결제 인센티브의 유효기간을 올해 말...
- 연령·소득 무관 모든 여성에 생리대 지원…주민센터·도서관 비치(종합) || ... 대상으로 월 1만 4000원 상당의 생리용품 구매 이용권을 지원해 왔다. 일부 지자체는 지역화폐 지원이나 생리대 자판기를 운영하고 교육청별로 학교 보건실을 통해 생리대를 지원하는...
- 대통령실 오는 19일 국무회의에 2차 추경안 상정 || ... 원 안팎의 규모가 될 전망이며, 소비 진작 등 내수 활성화를 위한 민생회복지원금 지급과 지역화폐 지원책 등이 담길 것으로 보입니다....
- 이재명 외교참사 책임 묻겠다…개헌특위 제안 || ...소득, 아동기본소득 등 약자들을 위한 기본 정책이 필요하다고 했습니다.  그러면서 정부가 지역화폐 예산을 전액 삭감한 것은 도저히 이해할 수 없다며, 민주당이 최선을 다해 이를 막...
- 동백전 QR결제 하세요…부산시 한 달간 이벤트 || ... 한다고 23일 밝혔다. 소상공인이 부담하는 카드 결제 수수료는 0.25~1.5%이지만, 지역화폐 '동백전'을 QR 결제로 하면 소상공인의 카드 수수료 부담이 없다. 예를 들어 이...
- 육아맘·육아대디 광명으로 모여라…26일 투맘쇼 || ...위해 광명시가 마련한 기획 공연이기도 하다. 아이

- 판정 메모: 본문 1회 등장 기사에는 동백전·여민전·인천이음 등 지역화폐 운영 기사와, 포상금·출산지원금을 지역화폐로 지급한다는 부수적 언급 기사가 혼재
- 리드 400자 기준(R3)은 부수적 언급 기사를 다수 포함하여 채택하지 않음
- 잠정 채택: R4 (동의어 3종 기준, 제목 포함 또는 본문 2회 이상이면서 1000자당 1회 이상)
- 한계: 지역화폐 브랜드명(동백전·여민전 등)만 쓰고 일반명사를 1회만 쓴 기사는 누락 가능

## 6. 보일러플레이트 패턴

In [12]:
PATTERNS = {
    '무단전재/재배포 금지': r'무단\s*전재|재배포\s*금지',
    '이메일': r'[\w.+-]+@[\w-]+\.[\w.]+',
    '기자명(OOO 기자)': r'[가-힣]{2,4}\s?기자',
    '사진 설명([사진] 등)': r'\[사진|사진=|\(사진=|ⓒ',
    '저작권 문구(Copyright)': r'Copyright|저작권자',
    '관련기사/바로가기': r'관련\s*기사|바로가기|▶',
}
rows = []
for name, p in PATTERNS.items():
    m = df['본문전문'].str.contains(p, regex=True)
    rows.append((name, int(m.sum()), round(m.mean(), 3)))
bp = pd.DataFrame(rows, columns=['pattern', 'docs', 'ratio'])
SUMMARY['boilerplate'] = {r.pattern: r.docs for r in bp.itertuples()}
bp

,pattern,docs,ratio
0,무단전재/재배포 금지,41,0.001
1,이메일,0,0.000
2,기자명(OOO 기자),2498,0.068
3,사진 설명([사진] 등),0,0.000
4,저작권 문구(Copyright),79,0.002
5,관련기사/바로가기,82,0.002


In [13]:
# 본문 끝 40자 빈도: 반복되는 꼬리말 식별
tails = df['본문전문'].str.strip().str[-40:]
print(tails.value_counts().head(15).to_string())

본문전문
보고 싶다면?          기사 섹션 분류 안내 이 기사를 추천합니다    3383
했다.   기사를 더 보고 싶다면?          기사 섹션 분류 안내    1063
보' 검색  이메일    뉴스 홈페이지   bbs report write     879
      기사를 더 보고 싶다면?          기사 섹션 분류 안내     844
거리를 알려주세요. 이메일    카카오톡     사이트    b71afn     470
 '제보' 검색, 채널 추가   네이버, 유튜브에서 뉴스를 구독해주세요!     230
다.    기사를 더 보고 싶다면?          기사 섹션 분류 안내     224
 Biz는 여러분의 제보를 기다리고 있습니다. 홈페이지    9pghjn     219
이다.   기사를 더 보고 싶다면?          기사 섹션 분류 안내     190
알려주세요. 이메일    카카오톡    노컷뉴스 사이트    b71afn     142
있다.   기사를 더 보고 싶다면?          기사 섹션 분류 안내     134
혔다.   기사를 더 보고 싶다면?          기사 섹션 분류 안내     131
.     기사를 더 보고 싶다면?          기사 섹션 분류 안내     121
세요.   기사를 더 보고 싶다면?          기사 섹션 분류 안내     107
ite   기사를 더 보고 싶다면?          기사 섹션 분류 안내      86


In [14]:
# 꼬리말 제거 규칙 후보와 적용 대상 문서 수
TAIL_PATTERNS = {
    '기사 섹션 분류 안내': r'기사를 더 보고 싶다면\?.*$',
    '제보 안내': r'제보를 기다리고 있습니다.*$|제보하기\s+전화.*$|거리를 알려주세요\..*$',
    '구독 안내': r'네이버,\s*유튜브에서 뉴스를 구독해주세요.*$',
}
SUMMARY['tail_boilerplate'] = {}
for name, p in TAIL_PATTERNS.items():
    n = int(df['본문전문'].str.contains(p, regex=True).sum())
    SUMMARY['tail_boilerplate'][name] = n
    print(f'{name}: {n}건')

기사 섹션 분류 안내: 7311건


제보 안내: 1412건
구독 안내: 238건


## 7. 중복 실태 — 완전 중복과 near-dup

In [15]:
def norm(s):
    return re.sub(r'\s+', '', s)

df['body_hash'] = df['본문전문'].map(lambda s: hashlib.md5(norm(s).encode()).hexdigest())
df['title_norm'] = df['제목'].map(lambda s: re.sub(r'[^가-힣A-Za-z0-9]', '', s))
exact = {
    'url_dup': int(df['원문URL'].duplicated().sum()),
    'body_dup': int(df['body_hash'].duplicated().sum()),
    'title_dup': int(df['title_norm'].duplicated().sum()),
}
print(exact)
SUMMARY['exact_dup'] = exact

{'url_dup': 1, 'body_dup': 625, 'title_dup': 1496}


In [16]:
from datasketch import MinHash, MinHashLSH

NUM_PERM = 64
SHINGLE = 5

def minhash(text):
    t = norm(text)[:3000]
    m = MinHash(num_perm=NUM_PERM, seed=1)
    for i in range(max(1, len(t) - SHINGLE + 1)):
        m.update(t[i:i + SHINGLE].encode('utf-8'))
    return m

base = df[~df['body_hash'].duplicated()].copy()
base = base[base.body_len >= 50]
mhs = {}
lsh = MinHashLSH(threshold=0.5, num_perm=NUM_PERM)
for idx, text in zip(base.index, base['본문전문']):
    m = minhash(text)
    mhs[idx] = m
    lsh.insert(idx, m)
pairs = {}
for idx, m in mhs.items():
    for j in lsh.query(m):
        if j > idx:
            pairs[(idx, j)] = mhs[idx].jaccard(mhs[j])
print('완전중복 제거 후 문서:', len(base), ' 후보 쌍:', len(pairs))

완전중복 제거 후 문서: 36232  후보 쌍: 18947


In [17]:
def clusters_at(th):
    parent = {i: i for i in base.index}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for (a, b), s in pairs.items():
        if s >= th:
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    return pd.Series({i: find(i) for i in base.index})

sweep = []
for th in [0.5, 0.6, 0.7, 0.8, 0.9]:
    cl = clusters_at(th)
    sweep.append({'threshold': th, 'clusters': int(cl.nunique()), 'removed': int(len(cl) - cl.nunique())})
sweep = pd.DataFrame(sweep)
SUMMARY['neardup_sweep'] = sweep.to_dict('records')
print(sweep.to_string(index=False))
fig = px.line(sweep, x='threshold', y='clusters', markers=True, title='near-dup 임계값별 잔존 문서 수')
fig.show()

 threshold  clusters  removed
       0.5     29203     7029
       0.6     30796     5436
       0.7     32409     3823
       0.8     34154     2078
       0.9     35368      864


In [18]:
# 유사도 구간별 쌍 샘플: 임계값 결정을 위한 직접 판정 자료
pair_df = pd.DataFrame([(a, b, s) for (a, b), s in pairs.items()], columns=['a', 'b', 'sim'])
for lo, hi in [(0.5, 0.6), (0.6, 0.7), (0.7, 0.8), (0.8, 0.9)]:
    sub = pair_df[(pair_df.sim >= lo) & (pair_df.sim < hi)]
    print(f'==== [{lo}, {hi}) {len(sub)}쌍')
    for _, p in sub.sample(min(4, len(sub)), random_state=3).iterrows():
        A, B = df.loc[p.a], df.loc[p.b]
        print(f'  sim={p.sim:.2f}')
        print(f'   A {A["발행일"][:10]} {A["매체명"]} | {A["제목"][:60]}')
        print(f'   B {B["발행일"][:10]} {B["매체명"]} | {B["제목"][:60]}')

==== [0.5, 0.6) 4341쌍
  sim=0.59
   A 2024-11-21 뉴스1 | 김동연 경제정책 역주행 정부 직격…이재명 지역화폐 지속돼야(종합)
   B 2024-11-21 스포츠서울 | 김동연 “내년도 지역화폐 예산 1043억 원 편성했다”
  sim=0.52
   A 2024-07-17 머니투데이 | CU, 조폐공사와 팀코리아 팝업…국가대표 선수 '골드 포토카드' 판매
   B 2024-07-17 데일리안 | “금맥 캐는 편의점” BGF리테일, 한국조폐공사 전략적 제휴
  sim=0.59
   A 2025-06-19 데일리안 | 스마트도시 조성사업에 대구·전주 등 5개 지자체 선정
   B 2025-06-19 지디넷코리아 | 국토부, ‘2025 스마트도시 조성사업’…전주·안산·김천·김해·대구시 선정
  sim=0.50
   A 2024-05-20 경향신문 | 경기도, 경기지역화폐 부정유통 행위 합동 단속
   B 2024-05-20 데일리안 | 경기도, 경기지역화폐 '부정유통' 단속
==== [0.6, 0.7) 3660쌍
  sim=0.66
   A 2024-02-01 더팩트 | 임병택 시흥시장 시흥화폐, 지역경제·공동체 위한 도구
   B 2024-02-01 디지털타임스 | `경제도시 시흥시`, 시흥화폐 `시루`로 소상공인 활력 충전,,,中企 특례보증도 최대 3억 지원
  sim=0.69
   A 2022-10-09 스포츠서울 | 경기도,박물관 등 35군데 문화시설 이용료 할인...17~23일 까지
   B 2022-10-09 데일리안 | 경기도민의 날 맞아 문화시설 이용 최대 1만5000원 할인
  sim=0.69
   A 2025-12-17 디지털데일리 | 하나은행∙하나카드, 지역 상권 활성화를 위해 대전광역시와 맞손…데이터 교류 활성화
   B 2025-12-17 아시아경제 | 하나은행·카드, 대전광역시와 지역상권활성화 위한 데이터 협력 나서
  sim=0.62
   A 2021-12-28 헤럴드경제 | 성남시 지역화폐…지자체 우

- 판정 메모: [0.5, 0.6) 구간은 같은 사건을 다른 관점·분량으로 다룬 기사가 다수 → 유지 대상
- [0.6, 0.7) 구간은 동일 보도자료 기반 재작성과 동일 사건 별도 보도가 혼재
- 0.7 이상 구간 샘플은 동일 보도자료의 매체별 재배포본 → 잠정 임계값 0.7

## 8. 전처리 퍼널 (잠정 기준)

In [19]:
TH_NEAR = 0.7
rel = cands['R4: 동의어 제목 or (본문>=2 and 1000자당>=1)']
cl = clusters_at(TH_NEAR)
funnel = {}
funnel['0_raw'] = len(df)
s1 = df[rel]
funnel['1_relevance(R4)'] = len(s1)
s2 = s1[~s1['body_hash'].duplicated() & ~s1['원문URL'].duplicated()]
s2 = s2[s2.index.isin(cl.index)]
funnel['2_exact_dedup'] = len(s2)
s3 = s2.assign(cluster=cl.reindex(s2.index)).sort_values('body_len', ascending=False).drop_duplicates('cluster')
funnel[f'3_neardup(>={TH_NEAR})'] = len(s3)
s4 = s3[s3.body_len >= 200]
funnel['4_len>=200'] = len(s4)
SUMMARY['funnel'] = {k: int(v) for k, v in funnel.items()}
SUMMARY['funnel_by_year'] = {int(k): int(v) for k, v in s4['year'].value_counts().sort_index().items()}
print(pd.Series(funnel).to_string())
print(s4['year'].value_counts().sort_index().to_string())
fig = px.funnel(x=list(funnel.values()), y=list(funnel.keys()), title='전처리 단계별 잔존 문서 수')
fig.show()

0_raw               36888
1_relevance(R4)     19322
2_exact_dedup       18896
3_neardup(>=0.7)    16946
4_len>=200          16760
year
2021    1250
2022    3170
2023    2043
2024    2466
2025    5023
2026    2808


## 9. 직접 정독용 샘플 (연도별 층화 20건)

In [20]:
samp = s4.groupby('year', group_keys=False).apply(lambda g: g.sample(min(4, len(g)), random_state=7))
for _, r in samp.iterrows():
    print('=' * 100)
    print(f'{r["발행일"][:10]} | {r["매체명"]} | len={r.body_len} | kw={r.key_cnt}')
    print('제목:', r['제목'])
    print(r['본문전문'][:700])

2021-12-01 | 뉴스1 | len=1376 | kw=6
제목: 여야, '경항모' 이견에 내년 예산 합의 불발…2일 추가 협의(종합)
與경항모 예산 반영 주장하며 막판 변수 발생 野 "알박기식" 반발 내년 예산 607조원, 감액 5.6조원엔 합의 지역화폐 손실보상도 입장차    여야가 1일 내년도 예산안을 정부안보다 3조원가량 순증한 607조원 규모로 처리하기로 잠정 합의했지만, 해군의 경항공모함(경항모) 도입 사업을 두고 막판 진통을 겪어 최종 합의는 불발됐다. 더불어민주당과 국민의힘은 이날 오후 2시부터 오후 10시까지 8시간 동안 원내수석부대표 등이 참석한 가운데 내년도 예산안 수정안에 대해 논의했지만 처리 여부와 총액, 감액 규모 등 일부만 합의하고 최종 수정안 마련에는 실패했다. 한병도 민주당 원내수석부대표는 이날 국회에서 진행된 예산 협의 회동 중간 기자들과 만나 "예산에 대해 (여야) 합의가 거의 이뤄졌는데 다른 데서 문제가 생겼다"며 "야당에서 경항모와 관련된 것을 문제제기하고 있어 당혹스러운 상황"이라고 전했다. 한 수석부대표는 "민주당은 경항모 설계를 해서 진행하려고 한다"며 "국방과 관련 우리나라의 국력을 생각하면 경항모가 어느 정부를 떠나서 굉장히 중요한 사업이라고 생각하기 때문에 반영을 꼭 시켜야겠다고 생각하는데 거기에 (야당 측이)반대하고 있다"고 말했다. 국회 예산결산특별위원회 야당 간사인 이만희 국민의힘 의원은 "경항모 사업은 내년에 편성되는 금액의 문제가 아니고 사업 자체만 수십조원에 이르는 대규모 사업"이라며 "그런 사업을 속칭 알박
2021-11-13 | 매일경제 | len=2055 | kw=3
제목: [랭킹쇼] 李·추진력 vs 尹·정의
제 21대 대선의 대진표가 완성됐다. 이재명 더불어민주당 후보와 윤석열 국민의힘 후보간 치열한 경쟁의 막이 오른 것이다. 각 후보가 내세우는 강점은 대표적으로 추진력과 정의다. 그러나 유권자들의 등을 돌리게 만드는 비호감도도 만만치 않다. 그들의 장단점을 정리해본다. 1. '이재명은 합니

## 10. 요약 저장

In [21]:
with open(os.path.join(OUT_DIR, 'eda_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(SUMMARY, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps(SUMMARY, ensure_ascii=False, indent=1, default=str)[:3000])

{
 "raw_rows": 36888,
 "empty_body": 41,
 "by_year": {
  "2021": 2622,
  "2022": 6986,
  "2023": 4471,
  "2024": 5304,
  "2025": 10694,
  "2026": 6811
 },
 "peak_months": {
  "2025-06": 1911,
  "2025-07": 1508,
  "2021-11": 1255,
  "2025-01": 1082,
  "2026-03": 1061,
  "2021-12": 1037,
  "2022-12": 1016,
  "2025-05": 928
 },
 "n_media": 96,
 "wire_share": 0.2357,
 "top_media": {
  "뉴시스": 3237,
  "뉴스1": 2804,
  "연합뉴스": 2654,
  "KBS": 1463,
  "파이낸셜뉴스": 1241,
  "아시아경제": 1177,
  "머니투데이": 1100,
  "더팩트": 1084,
  "헤럴드경제": 1048,
  "노컷뉴스": 1007
 },
 "body_len_quantiles": {
  "0.01": 145,
  "0.05": 355,
  "0.25": 664,
  "0.5": 969,
  "0.75": 1458,
  "0.95": 2745,
  "0.99": 6154
 },
 "short_lt200": 605,
 "long_gt10000": 157,
 "relevance_candidates": {
  "A: 제목 포함": 8062,
  "B: 제목 포함 or 본문>=2": 18515,
  "C: 제목 포함 or 본문>=3": 12914,
  "R3: 동의어 제목 or 리드400 or (본문>=2 and 1000자당>=1)": 26218,
  "R4: 동의어 제목 or (본문>=2 and 1000자당>=1)": 19322
 },
 "boilerplate": {
  "무단전재/재배포 금지": 41,
  "이메일": 0,
  "기자명(OOO